# Lower-Level API: MinHash + LSHForest + Layout

Use this notebook when you want the original staged TMAP workflow instead of the estimator.

This path is useful when you want direct control over:

- MinHash settings
- LSHForest construction
- kNN graph inspection
- layout from the forest or from a precomputed graph

For the shortest workflow, use `TMAP(...).fit(X)` instead.


## 1. Load Molecules And Build Fingerprints

We use the same molecule dataset as the main chemistry tutorial so the two APIs are easy to compare.


In [1]:
import pandas as pd

from tmap.utils import fingerprints_from_smiles, molecular_properties

df = pd.read_csv("../examples/cluster_65053.csv", nrows=2000)
smiles = df["smiles"].tolist()
props = molecular_properties(smiles, properties=["mw", "logp", "n_rings"])
fps = fingerprints_from_smiles(smiles, fp_type="morgan", radius=2, n_bits=2048)

print(f"Loaded {len(smiles):,} molecules")
print(f"Fingerprint matrix: {fps.shape}")


  [Props] 2,000 done, 0 invalid
Loaded 2,000 molecules
Fingerprint matrix: (2000, 2048)


## 2. MinHash

`batch_from_binary_array()` is the usual entry point for dense binary fingerprints.
See `02_minhash_deep_dive.ipynb` for the full set of `from_*` and `batch_from_*` methods.


In [2]:
from tmap import MinHash

mh = MinHash(num_perm=128, seed=42)
signatures = mh.batch_from_binary_array(fps)
single_signature = mh.from_binary_array(fps[0])

print(signatures.shape)
print(single_signature.shape)


(2000, 128)
(128,)


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


## 3. Build LSHForest

This is the main lower-level index for the Jaccard workflow.


In [3]:
from tmap import LSHForest

lsh = LSHForest(d=128, l=64)
lsh.batch_add(signatures)
lsh.index()

knn = lsh.get_knn_graph(k=20, kc=50)
print(knn.indices.shape)
print(knn.distances.shape)


(2000, 20)
(2000, 20)


## 4. Compute The Layout Directly From The Forest

`layout_from_lsh_forest()` is the closest match to the classic TMAP pipeline.


In [4]:
from tmap.layout import LayoutConfig, layout_from_lsh_forest

cfg = LayoutConfig()
cfg.k = 20
cfg.kc = 50
cfg.deterministic = True
cfg.seed = 42

x, y, s, t = layout_from_lsh_forest(lsh, cfg)
print(len(x), len(s))


2000 1999


## 5. Alternative: Layout From A Precomputed kNN Graph

Use this when you want to inspect or modify the graph first.


In [5]:
from tmap.layout import layout_from_knn_graph

x2, y2, s2, t2 = layout_from_knn_graph(knn, cfg)
print(len(x2), len(s2))


2000 1999


## 6. Build A Visualization Manually

Here we create a `TmapViz` object without using the estimator.


In [6]:
from tmap.visualization import TmapViz

viz = TmapViz()
viz.title = "Legacy LSH Pipeline"
viz.set_points(x, y)
viz.set_edges(s, t)
viz.add_smiles(smiles)
viz.add_color_layout("MW", props["mw"].tolist(), color="viridis")
viz.add_color_layout("LogP", props["logp"].tolist(), color="plasma")
viz.add_color_layout("Ring Count", props["n_rings"].tolist(), categorical=True, color="tab10")

path = viz.write_html("legacy_lsh_pipeline.html")
print(path)


legacy_lsh_pipeline.html


## 7. When To Use This API

Use the lower-level pipeline when:

- you want direct MinHash control
- you want to inspect the LSH forest or the kNN graph
- you want to lay out an externally built graph

Use the estimator when you just want to pass data and get a map.
